# IV_06 — ML: clasificación de fallas

## 1. Objetivo

Entrenar un clasificador para anticipar fallas en PUMP101 usando variables de condición multivariable.

## 2. Concepto

**Clasificación binaria:** predecir `falla=1` (inminente) vs `falla=0` (normal).

**Costo industrial:**
- Falso positivo → parada innecesaria, pérdida de producción.
- Falso negativo → falla no detectada, daño mayor.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MOD_DIR = Path.cwd()
os.chdir(MOD_DIR)
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR = MOD_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "dataset_predictivo.csv"

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
features = [
    "PUMP101.BEARING_TEMP", "PUMP101.VIBRATION_RMS",
    "PUMP101.DISCHARGE_PRESS", "PUMP101.MOTOR_CURRENT",
    "vib_media_24h", "temp_pendiente_24h", "presion_pct",
]
X = df[features]
y = df["falla"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Normal", "Falla"]))


## 3. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cm = confusion_matrix(y_test, y_pred)
axes[0].imshow(cm, cmap="Blues")
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Normal", "Falla"]); axes[0].set_yticklabels(["Normal", "Falla"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center")
axes[0].set_title("Matriz de confusión")

imp = pd.Series(modelo.feature_importances_, index=features).sort_values()
imp.plot(kind="barh", ax=axes[1], color="teal")
axes[1].set_title("Importancia de features")

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].set_xlabel("")
axes[2].plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.3f}")
axes[2].plot([0, 1], [0, 1], "k--")
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("Curva ROC")
axes[2].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ml_clasificacion.png", dpi=150)
print("Gráfico guardado.")


## 4. Interpretación para mantenimiento

La vibración RMS y su media móvil suelen ser los predictores dominantes. Priorizar monitoreo de rodamiento cuando importancia > 20%.

## 5. Ejercicio práctico

Si el recall de falla es 0.85, ¿qué significa para el planificador de mantenimiento?

## 6. Resumen y siguiente paso

- Random Forest maneja bien datos tabulares industriales.
- Evaluar recall vs precisión según costo de parada.
- Importancia de features guía inversiones en sensores.

**Siguiente:** `IV_07_anomalias_y_rul.ipynb`